This code allows us to trace all the formulas in a given worksheet, including the interdependencies between different cells, and display the formulas in a structured, hierarchical format. It handles both individual cell references and ranges, providing detailed insights into how formulas are linked across the spreadsheet.

In [ ]:
pip install openpyxl 

In [ ]:
pip install networkx

In [ ]:
pip install matplotlib

In [ ]:
pip install xlwings 

In [ ]:
import openpyxl
import xlwings as xw
import pandas as pd
import shutil
import io
import os
import pandas as pd
import time
import re

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
user_profile = os.getenv('USERPROFILE') ## Retrieving user profile directory from 'USERPROFILE' environment variable.

In [ ]:
# Constructing the base directory path using the user profile directory.
base_dir = os.path.join(
    user_profile, 
    'California Department of Transportation',
    'DOT HQ PMP Cal B C Update - General',
    'Testbed'
)

In [ ]:
sketch_file_path = os.path.join(base_dir, 'Input', 'cal-bc-8-1-sketch-a11y.xlsm')

In [ ]:
# This function generates a list of cell references between a given start and end cell range (inclusive).

def cell_range_to_list(start, end):
    # Convert the start and end cells of a range to a list of individual cells
    start_col, start_row = re.match(r"([A-Za-z]+)(\d+)", start).groups()
    end_col, end_row = re.match(r"([A-Za-z]+)(\d+)", end).groups()

    # Convert column letters to column numbers
    start_col_num = openpyxl.utils.column_index_from_string(start_col)
    end_col_num = openpyxl.utils.column_index_from_string(end_col)

    start_row = int(start_row)
    end_row = int(end_row)

    # Generate all cell references between the start and end points
    cell_references = []
    for col in range(start_col_num, end_col_num + 1):
        for row in range(start_row, end_row + 1):
            cell_ref = openpyxl.utils.get_column_letter(col) + str(row)
            cell_references.append(cell_ref)

    return cell_references


In [ ]:
# This function extracts all cell references (both single cells and ranges) from a given formula.

def extract_cell_references(formula):
    # Regex pattern to match cell references (single cells or ranges)
    pattern = r'([A-Za-z]+\d+)(?::([A-Za-z]+\d+))?'  # Matches cells and ranges like A1 or A1:B10
    
    matches = re.findall(pattern, formula)
    
    all_references = []
    for start, end in matches:
        if end:  # This is a range
            all_references.extend(cell_range_to_list(start, end))
        else:  # Single cell reference
            all_references.append(start)
    
    return all_references





In [ ]:
def trace_formula_references(cell, level=0, wb=None, visited=None, formula_data=None):
    if visited is None:
        visited = set()  # Keeps track of already visited cells to avoid infinite recursion
    
    if formula_data is None:
        formula_data = {}
    
    if cell.address in visited:
        return formula_data
    
    visited.add(cell.address)
    
    formula = cell.formula
    
    # Add formula or value to formula_data with indentation based on depth
    indent = " " * (level * 2)
    if formula:
        formula_data[cell.address] = f"{indent}{formula}"
    else:
        formula_data[cell.address] = f"{indent}{cell.value if cell.value else 'No value'}"
    
    # Extract all cell references from the formula
    references = extract_cell_references(formula) if formula else []
    
    # Trace each referenced cell
    for ref in references:
        try:
            if '!' in ref:  # Handle cross-sheet references
                sheet_name, cell_ref = ref.split('!')
                sheet = wb.sheets[sheet_name]
            else:
                sheet = cell.sheet
                cell_ref = ref

            if ":" in cell_ref:  # Handle ranges
                start_cell, end_cell = cell_ref.split(":")
                start_cell = sheet.range(start_cell)
                end_cell = sheet.range(end_cell)
                
                # Loop through all cells in the range
                for row in range(start_cell.row, end_cell.row + 1):
                    for col in range(start_cell.column, end_cell.column + 1):
                        cell_in_range = sheet.range(row, col)
                        if cell_in_range.formula:
                            formula_data[cell_in_range.address] = f"{indent}  {cell_in_range.formula}"
                        elif cell_in_range.value is None or cell_in_range.value == '':
                            formula_data[cell_in_range.address] = f"{indent}  No value"
                        else:
                            formula_data[cell_in_range.address] = f"{indent}  {cell_in_range.value}"

                        # Recursively trace formulas inside the range
                        if cell_in_range.formula and cell_in_range.address not in visited:
                            trace_formula_references(cell_in_range, level + 1, wb, visited, formula_data)
            else:
                referenced_cell = sheet.range(cell_ref)
                if referenced_cell.formula:
                    formula_data[referenced_cell.address] = f"{indent}  {referenced_cell.formula}"
                elif referenced_cell.value is None or referenced_cell.value == '':
                    formula_data[referenced_cell.address] = f"{indent}  No value"
                else:
                    formula_data[referenced_cell.address] = f"{indent}  {referenced_cell.value}"

                # Recursively trace formulas of referenced cells
                if referenced_cell.formula and referenced_cell.address not in visited:
                    trace_formula_references(referenced_cell, level + 1, wb, visited, formula_data)
        except Exception as e:
            print(f"Could not trace {ref}: {str(e)}")
    
    return formula_data


In [ ]:
# Open Workbook
app = xw.App(visible=False)
wb = app.books.open(sketch_file_path)

In [ ]:
# Select specific sheet and cell to trace formulas 
ws = wb.sheets[3]
cell = ws.range('V52')

In [ ]:
formula_data = trace_formula_references(cell, level=0, wb=wb)

# Print the formatted output (descending order with indentation)
for cell, formula in formula_data.items():
    i = 0
    while formula[i] == ' ':
        i+=1
        
    print(f"{' ' * i}{cell}: {formula.strip()}")